In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
bureau = pd.read_csv(
    "data/raw/home_credit/bureau.csv",
)

In [ ]:
bureau.shape

In [ ]:
bureau.head()

In [ ]:
bureau.info()

In [ ]:
bureau.describe()

In [ ]:
bureau.isnull().sum().sort_values(
    ascending=False
)

In [ ]:
print(
    "Total bureau records:",
    bureau.shape[0]
)

print(
    "Unique customers:",
    bureau["SK_ID_CURR"].nunique()
)

print(
    "Unique bureau records:",
    bureau["SK_ID_BUREAU"].nunique()
)

print(
    "Duplicated bureau IDs:",
    bureau["SK_ID_BUREAU"].duplicated().sum()
)

In [ ]:
bureau_records_per_customer = (
    bureau.groupby("SK_ID_CURR")
    .size()
)

bureau_records_per_customer.describe(
    percentiles=[
        0.25,
        0.5,
        0.75,
        0.9,
        0.95,
        0.99
    ]
)

In [ ]:
bureau_records_per_customer.sort_values(
    ascending=False
).head(10)

In [ ]:
bureau["CREDIT_ACTIVE"].value_counts(
    dropna=False
)

In [ ]:
bureau["CREDIT_CURRENCY"].value_counts(
    dropna=False
)

In [ ]:
bureau_missing_summary = pd.DataFrame({
    "MISSING_COUNT": bureau.isna().sum(),
    "MISSING_RATE": (
        bureau.isna().mean() * 100
    )
}).sort_values(
    by="MISSING_RATE",
    ascending=False
)

bureau_missing_summary

In [ ]:
#每个客户一共有多少条外部历史信用记录

In [ ]:
bureau_count_features = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        BUREAU_RECORD_COUNT=(
            "SK_ID_BUREAU",
            "count"
        )
    )
    .reset_index()
)

bureau_count_features.head()

In [ ]:
#当前活跃账户数

In [ ]:
bureau["IS_ACTIVE"] = (
    bureau["CREDIT_ACTIVE"] == "Active"
).astype("int8")

In [ ]:
active_credit_feature = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        ACTIVE_CREDIT_COUNT=(
            "IS_ACTIVE",
            "sum"
        )
    )
    .reset_index()
)

active_credit_feature.head()

In [ ]:
#已结清账户数

In [ ]:
bureau["IS_CLOSED"] = (
    bureau["CREDIT_ACTIVE"] == "Closed"
).astype("int8")

In [ ]:
closed_credit_feature = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        CLOSED_CREDIT_COUNT=(
            "IS_CLOSED",
            "sum"
        )
    )
    .reset_index()
)

closed_credit_feature.head()

In [ ]:
#坏账户数

In [ ]:
bureau["IS_BAD_DEBT"] = (
    bureau["CREDIT_ACTIVE"] == "Bad debt"
).astype("int8")

In [ ]:
bad_debt_feature = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        BAD_DEBT_COUNT=(
            "IS_BAD_DEBT",
            "sum"
        )
    )
    .reset_index()
)

bad_debt_feature.head()

In [ ]:
#已转售账户数，有些债务会卖给催收公司

In [ ]:
bureau["IS_SOLD"] = (
    bureau["CREDIT_ACTIVE"] == "Sold"
).astype("int8")

In [ ]:
sold_credit_feature = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        SOLD_CREDIT_COUNT=(
            "IS_SOLD",
            "sum"
        )
    )
    .reset_index()
)

sold_credit_feature.head()

In [ ]:
#把上面建的特征工程合并

In [ ]:
bureau_account_features = (
    bureau_count_features
    .merge(
        active_credit_feature,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        closed_credit_feature,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        bad_debt_feature,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        sold_credit_feature,
        on="SK_ID_CURR",
        how="left"
    )
)

bureau_account_features.head()

In [ ]:
##构建与贷款金额相关的特征工程

In [ ]:
bureau_amount_features = (
    bureau.groupby("SK_ID_CURR")
    .agg(

        TOTAL_CREDIT_SUM=(
            "AMT_CREDIT_SUM",
            "sum"
        ),

        AVERAGE_CREDIT_SUM=(
            "AMT_CREDIT_SUM",
            "mean"
        ),

        MAX_CREDIT_SUM=(
            "AMT_CREDIT_SUM",
            "max"
        ),

        TOTAL_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum"
        ),

        AVERAGE_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "mean"
        ),

        TOTAL_OVERDUE_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum"
        ),

        MAX_OVERDUE_AMOUNT=(
            "AMT_CREDIT_MAX_OVERDUE",
            "max"
        ),

        TOTAL_CREDIT_LIMIT=(
            "AMT_CREDIT_SUM_LIMIT",
            "sum"
        )

    )
    .reset_index()
)

bureau_amount_features.head()

In [ ]:
##构建时间相关的特征工程

In [ ]:
bureau_time_features = (
    bureau.groupby("SK_ID_CURR")
    .agg(

        LATEST_CREDIT_DAYS=(
            "DAYS_CREDIT",
            "max"
        ),

        OLDEST_CREDIT_DAYS=(
            "DAYS_CREDIT",
            "min"
        ),

        AVERAGE_CREDIT_DAYS=(
            "DAYS_CREDIT",
            "mean"
        ),

        MAX_CREDIT_OVERDUE_DAYS=(
            "CREDIT_DAY_OVERDUE",
            "max"
        ),

        AVERAGE_CREDIT_OVERDUE_DAYS=(
            "CREDIT_DAY_OVERDUE",
            "mean"
        ),

        LATEST_CREDIT_UPDATE_DAYS=(
            "DAYS_CREDIT_UPDATE",
            "max"
        ),

        AVERAGE_CREDIT_UPDATE_DAYS=(
            "DAYS_CREDIT_UPDATE",
            "mean"
        )

    )
    .reset_index()
)

bureau_time_features.head()

In [ ]:
bureau_time_features.describe()

In [ ]:
# 构建信用类型的特征

In [ ]:
bureau["IS_CONSUMER_CREDIT"] = (
    bureau["CREDIT_TYPE"] == "Consumer credit"
).astype("int8")

bureau["IS_CREDIT_CARD"] = (
    bureau["CREDIT_TYPE"] == "Credit card"
).astype("int8")

bureau["IS_MORTGAGE"] = (
    bureau["CREDIT_TYPE"] == "Mortgage"
).astype("int8")

bureau["IS_CAR_LOAN"] = (
    bureau["CREDIT_TYPE"] == "Car loan"
).astype("int8")

bureau["IS_MICROLOAN"] = (
    bureau["CREDIT_TYPE"] == "Microloan"
).astype("int8")

In [ ]:
bureau_type_features = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        CONSUMER_CREDIT_COUNT=(
            "IS_CONSUMER_CREDIT",
            "sum"
        ),

        CREDIT_CARD_COUNT=(
            "IS_CREDIT_CARD",
            "sum"
        ),

        MORTGAGE_COUNT=(
            "IS_MORTGAGE",
            "sum"
        ),

        CAR_LOAN_COUNT=(
            "IS_CAR_LOAN",
            "sum"
        ),

        MICROLOAN_COUNT=(
            "IS_MICROLOAN",
            "sum"
        )
    )
    .reset_index()
)

bureau_type_features.head()

In [ ]:
bureau_type_features.describe()

In [ ]:
bureau_features = (
    bureau_account_features
    .merge(
        bureau_amount_features,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        bureau_time_features,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        bureau_type_features,
        on="SK_ID_CURR",
        how="left"
    )
)

In [ ]:
bureau_features.shape

In [ ]:
bureau_features["SK_ID_CURR"].duplicated().sum()

In [ ]:
bureau_features.isna().sum().sort_values(ascending=False)

In [ ]:
bureau_features.columns.tolist()

#与一开始的主表进行merge

In [ ]:
bureau_features.to_parquet(
    "data/processed/bureau_features.parquet",
    index=False
)

In [ ]:
import pandas as pd

application_train = pd.read_parquet(
    "data/processed/application_features.parquet"
)

application_train.shape

In [ ]:
bureau_features = (
    bureau_account_features
    .merge(
        bureau_amount_features,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        bureau_time_features,
        on="SK_ID_CURR",
        how="left"
    )
    .merge(
        bureau_type_features,
        on="SK_ID_CURR",
        how="left"
    )
)

bureau_features.head()

In [ ]:
bureau_features.shape

In [ ]:
bureau_features.shape[0] == bureau["SK_ID_CURR"].nunique()

In [ ]:
bureau_features.shape[0] == bureau["SK_ID_CURR"].nunique()

In [ ]:
application_train = application_train.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

application_train.shape

In [ ]:
application_rows_before = application_train.shape[0]

print("主表 Merge 前：", application_train.shape)
print("bureau_features：", bureau_features.shape)

In [ ]:
#检查merge 后行数是否和主表保持一致
print("Merge 后：", application_train.shape)

print(
    "行数是否保持不变：",
    application_train.shape[0] == application_rows_before
)

print(
    "重复客户数量：",
    application_train["SK_ID_CURR"].duplicated().sum()
)

In [ ]:
#验证bureau特征是否成功加入
bureau_feature_columns = [
    col
    for col in bureau_features.columns
    if col != "SK_ID_CURR"
]

bureau_feature_columns

In [ ]:
application_train["BUREAU_RECORD_COUNT"].isna().sum()

In [ ]:
application_train["BUREAU_RECORD_COUNT"].isna().mean()

In [ ]:
application_train["HAS_BUREAU_HISTORY"] = (
    application_train["BUREAU_RECORD_COUNT"]
    .notna()
    .astype("int8")
)

In [ ]:
application_train[
    "HAS_BUREAU_HISTORY"
].value_counts()

In [ ]:
bureau_account_columns = [
    col
    for col in bureau_account_features.columns
    if col != "SK_ID_CURR"
]

bureau_account_columns

In [ ]:
application_train["BUREAU_RECORD_COUNT"] = (
    application_train["BUREAU_RECORD_COUNT"]
    .fillna(0)
)

In [ ]:
application_train[
    bureau_account_columns
].isna().sum()